# PARASYTE - LFM Training for CDP

## Fine-tuning LFM 2.5-1.2B to master Chrome DevTools Protocol

This notebook trains a model to become an expert in CDP commands.

In [ ]:
# @title 1. Install dependencies
!pip install -q transformers peft datasets accelerate bitsandbytes
!pip install -q unsloth
!pip install -q httpx 2>/dev/null || true

In [ ]:
# @title 2. Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')

In [ ]:
# @title 3. Clone repo and load dataset
!git clone https://github.com/R0B0WARRI0R/Parasyte.git 2>/dev/null || echo 'Repo already cloned'
import json
import os

# Load multilingual dataset
dataset_path = 'Parasyte/data/cdp_training_multilang.json'
if os.path.exists(dataset_path):
    with open(dataset_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    print('Dataset loaded:', len(raw_data), 'examples')
    print('Languages:', set(d['language'] for d in raw_data))
    print('Commands:', set(d['cdp_command'] for d in raw_data))
else:
    print('Dataset not found')

In [ ]:
# @title 4. Format dataset for training
from datasets import Dataset

def format_example(example):
    text = 'Eres un experto en DevTools Protocol.\n'
    text += 'Instruction: ' + example['instruction'] + '\n'
    text += 'Language: ' + example['language'] + '\n'
    text += 'Command: ' + example['cdp_command'] + '\n'
    text += 'Params: ' + json.dumps(example['cdp_params'])
    return {'text': text}

formatted = [format_example(ex) for ex in raw_data]
dataset = Dataset.from_list(formatted)
print('Dataset formatted:', len(dataset), 'examples')
print('Example:', dataset[0]['text'][:200])

In [ ]:
# @title 5. Load base model
from unsloth import FastLanguageModel

model_name = 'liquid/lfm-2.5-1.2b-instruct'
max_seq_length = 1024
dtype = None
load_in_4bit = True

print('Loading model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print('Model loaded:', model.num_parameters(), 'parameters')

In [ ]:
# @title 6. Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('LoRA applied. Trainable params:', trainable, '(', round(trainable/model.num_parameters()*100, 1), '%)')

In [ ]:
# @title 7. Tokenize dataset
def tokenize_function(examples):
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=max_seq_length,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

tokenized = dataset.map(tokenize_function, batched=True, remove_columns=['text'])
avg_tokens = sum(len(x['input_ids']) for x in tokenized) / len(tokenized)
print('Dataset tokenized:', len(tokenized), 'examples')
print('Average tokens:', round(avg_tokens))

In [ ]:
# @title 8. Configure training
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    output_dir='outputs',
    report_to='none',
)

print('Training config ready')

In [ ]:
# @title 9. Train
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

steps = len(tokenized) // (4 * 4) * 3
print('Starting training...')
print('Total steps:', steps)
trainer.train()

In [ ]:
# @title 10. Save model
model.save_pretrained('parasyte_cdp_expert')
tokenizer.save_pretrained('parasyte_cdp_expert')
print('Model saved: parasyte_cdp_expert/')

In [ ]:
# @title 11. Test model
FastLanguageModel.for_inference(model)

test_instructions = [
    'Cuanta memoria usa JavaScript?',
    'How much memory is JavaScript using?',
    'Combien de memoire utilise JavaScript?',
    'Wie viel Speicher nutzt JavaScript?',
]

for instr in test_instructions:
    prompt = 'Eres un experto en DevTools Protocol.\n'
    prompt += 'Instruction: ' + instr + '\n'
    prompt += 'Command:'
    
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=32, use_cache=True)
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    print('Question:', instr)
    print('Response:', response)
    print()

---
## Next Steps

1. **Upload to HuggingFace**: `model.push_to_hub('your-user/parasyte-cdp-expert')`
2. **Export to GGUF**: See `export_gguf.ipynb`
3. **Integrate with Parasyte**: Connect fine-tuned model to CDP bridge

## Resources

- [Unsloth](https://github.com/unslothai/unsloth)
- [PEFT](https://huggingface.co/docs/peft)
- [CDP Protocol](https://chromedevtools.github.io/devtools-protocol/)
- [Liquid AI](https://liquid.ai/)